In [ ]:
# YOLO Tracking 기능 정리
#
# Tracking은 영상에서 객체를 탐지한 뒤,
# 같은 객체가 다음 프레임에서도 계속 같은 객체인지 추적하는 기능이다.
#
# 일반 Detection:
# - 각 프레임마다 객체를 따로 탐지함
# - 이전 프레임의 객체와 현재 프레임의 객체를 연결하지 않음
#
# Tracking:
# - 객체를 탐지한 뒤 고유 ID를 부여함
# - 같은 객체가 움직여도 같은 ID로 계속 따라감
# - 영상 속 객체의 이동 경로, 등장 시간, 사라지는 시점을 확인할 수 있음
#
# 예시:
# - 사람 A -> id 1
# - 사람 B -> id 2
# - 다음 프레임에서도 같은 사람이라면 같은 id 유지
#
# Tracking에서 중요한 값:
# - box 좌표: 객체 위치
# - class: 객체 종류
# - confidence: 탐지 신뢰도
# - track id: 추적 중인 객체의 고유 번호
#
# 활용 예시:
# - 사람 수 세기
# - 차량 추적
# - 객체 이동 경로 분석
# - 특정 객체가 화면에 얼마나 오래 있었는지 확인
#
# 정리:
# Detection은 "무엇이 어디에 있는가"를 찾는 기능이고,
# Tracking은 "같은 객체가 어디로 이동하는가"를 따라가는 기능이다.

# YOLO Tracking에서 사용할 수 있는 대표 tracker 설정 파일
#
# 1. botsort.yaml
# - BoT-SORT 알고리즘 사용
# - 객체의 위치 정보 + 움직임 정보를 이용해서 추적
# - 경우에 따라 ReID(객체 재식별) 정보도 활용 가능
# - 사람/차량처럼 계속 따라가야 하는 객체 추적에 많이 사용
#
# 2. bytetrack.yaml
# - ByteTrack 알고리즘 사용
# - confidence가 낮은 detection 결과도 일부 활용해서 추적을 이어감
# - 빠르고 실용적임
# - 객체가 잠깐 흐려지거나 confidence가 낮아져도 ID를 유지하는 데 도움됨
#
# 정리:
# - botsort.yaml  : 조금 더 정교한 추적 방식
# - bytetrack.yaml: 빠르고 안정적인 기본 추적 방식

In [ ]:
# Ultralytics YOLO를 사용한 다중 객체 추적
# Ultralytics YOLO 기반, 트래킹은 영상에서 객체 탐지 + ID를 부여해 계속 추적

import cv2
from ultralytics import YOLO

model = YOLO("yolo11n.pt")
video_path = "road_car.mp4"

# 비디오캡쳐 : 지정한 동영상 파일을 frame 단위로 읽음
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("동영상 열기 실패")
    exit()

while True:
    # ret : 읽기 성공 여부
    # frame : 실제 이미지 데이터
    ret, frame = cap.read()

    # 프레임을 읽지 못하는 경우, 루프 종료
    if not ret:
        break

    results = model.track(
        source=frame,               # 분석할 입력 이미지(현재 읽은 1장의 프레임)
        persist=True,               # 추적 여부 : True
        tracker="bytetrack.yaml",   # 객체 추적 알고리즘 - 현재 프레임과 이전 프레임을 비교해 id를 부여
        verbose = False,
        show=False                  # 탐지 추적 결과를 화면에 보여줌. (객체 인식 바운딩 박스, id 표시)
        )
    
    result = results[0]

    # box, id 모두 존재하는 경우
    if result.boxes is not None and result.boxes.id is not None:
        boxes = result.boxes.xyxy.cpu().numpy()
        ids = result.boxes.id.cpu().numpy().astype(int)
        class_ids = result.boxes.cls.cpu().numpy().astype(int)

        # 박스 좌표, 추적 id, 클래스 번호를 묶어 반복 처리
        for box, track_id, class_id in zip(boxes, ids, class_ids):
            x1, y1, x2, y2 = map(int, box)

            # ex) 2 = car
            class_name = result.names[class_id]
            print(f"id : {track_id}\tclass : {class_name}\tbox :", x1, y1, x2, y2)
    
    annotated_frame = result.plot()

    display_frame = cv2.resize(annotated_frame, (960, 540))

    cv2.imshow("yolo12 tracking", display_frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()